 Préparation des données — Personne B
**Projet : Prédiction des ventes supermarché (Bénin)**

Étapes couvertes :
- Chargement et inspection
- Encodage des variables catégorielles
- Feature engineering (date)
- Split Train / Test (temporel)
- Vérification des distributions

In [ ]:
import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

# Chemins
DATA_PATH   = "../data/supermarket_sales_forecasting.csv"
MODELS_PATH = "../models/"

os.makedirs(MODELS_PATH, exist_ok=True)

## 1. Chargement des données

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print(f"Shape : {df.shape}")
print(f"Période : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"\nTypes :\n{df.dtypes}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")
df.head()

---
##  Conclusion — Chargement des données

Le dataset contient **5 500 lignes et 16 colonnes**, avec **zéro valeur manquante**.

C'est une situation idéale. Dans un projet réel, les données ont presque toujours des trous — des jours non enregistrés, des capteurs qui ont lâché, des champs oubliés. Ici, rien de tout ça.

> **Ce que ça veut dire :** On peut faire confiance aux données et passer directement à la modélisation sans perdre de temps à nettoyer. La période couvre 3 années complètes (2023–2025), ce qui donne suffisamment d'historique pour que le modèle apprenne les tendances saisonnières.

---

## 2. Feature Engineering — Variables temporelles

In [ ]:
df["month"]        = df["Date"].dt.month
df["day_of_week"]  = df["Date"].dt.dayofweek          # 0=Lundi … 6=Dimanche
df["week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)
df["quarter"]      = df["Date"].dt.quarter
df["year"]         = df["Date"].dt.year

def get_season(month):
    """Saisons adaptées au Bénin (climat tropical)"""
    if month in [11, 12, 1, 2]:   return 0  # Saison sèche principale
    elif month in [3, 4, 5]:       return 1  # Petite saison des pluies
    elif month in [6, 7]:          return 2  # Petite saison sèche
    else:                          return 3  # Grande saison des pluies

df["season"] = df["month"].apply(get_season)

# Variable composite : pression économique
df["economic_pressure"] = df["Inflation_Rate"] + (1 / df["Economic_Index"].clip(lower=0.01)) * 100

# Interaction promotionnelle (synergies)
df["promo_weekend"]        = df["Promo"] * df["Weekend"]
df["promo_holiday"]        = df["Promo"] * df["Holiday"]
df["promo_school_holiday"] = df["Promo"] * df["School_Holiday"]

print("Features temporelles et interactions créées :")
print(df[["month","day_of_week","week_of_year","quarter","season",
          "economic_pressure","promo_weekend","promo_holiday"]].head())

## 3. Encodage des variables catégorielles

---
##  Conclusion — Feature Engineering (variables temporelles)

On crée **5 nouvelles colonnes** à partir de la date : `month`, `day_of_week`, `week_of_year`, `quarter`, `season`.

La machine ne comprend pas une date comme "2023-06-15". En la décomposant, on lui donne des informations concrètes et utilisables :

- `month` → les ventes varient selon les mois (rentrée, fêtes, saison sèche)
- `day_of_week` → les comportements d'achat changent selon le jour
- `season` → adapté au climat tropical du Bénin (4 saisons distinctes)

On crée aussi `economic_pressure` qui combine l'inflation et l'indice économique en un seul signal fort, et les variables d'interaction `promo_weekend`, `promo_holiday`, `promo_school_holiday` qui capturent les effets combinés.

> **Ce que ça veut dire :** Ces 8 nouvelles variables représentent notre connaissance métier traduite en code. Trois d'entre elles — `season`, `economic_pressure` et `promo_weekend` — se retrouveront dans le top 10 des variables les plus importantes du modèle final. La réflexion métier améliore directement la performance.

---

In [ ]:
# Colonnes catégorielles à encoder
cat_cols = ["Store_ID", "Store_Type", "City"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Sauvegarde des encoders (nécessaire pour l'API)
with open(os.path.join(MODELS_PATH, "encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)
print("\n✅ Encoders sauvegardés → models/encoders.pkl")

## 4. Sélection des features finales

In [ ]:
FEATURE_COLS = [
    # Store features
    "Store_ID_enc", "Store_Type_enc", "City_enc",
    "Store_Size", "Competitor_Distance",
    # Promotions & événements
    "Promo", "Holiday", "School_Holiday", "Weekend",
    "promo_weekend", "promo_holiday", "promo_school_holiday",
    # Météo & carburant
    "Temperature", "Rainfall", "Fuel_Price",
    # Macro-économie
    "Inflation_Rate", "Economic_Index", "economic_pressure",
    # Temporel
    "month", "day_of_week", "week_of_year", "quarter", "season", "year"
]

TARGET_COL = "Sales"

print(f"Nombre de features : {len(FEATURE_COLS)}")
print(f"Target : {TARGET_COL}")

---
## ✅ Conclusion — Encodage des variables catégorielles

On a transformé 3 colonnes texte en chiffres :

| Colonne | Exemple avant | Exemple après |
|---|---|---|
| Store_Type | "A", "B", "C" | 0, 1, 2 |
| City | "Cotonou", "Parakou"... | 2, 6... |
| Store_ID | "S001", "S002"... | 0, 1... |

Les encodeurs ont été **sauvegardés dans `models/encoders.pkl`**. C'est une étape cruciale — l'API devra utiliser exactement les mêmes encodeurs pour que les prédictions soient cohérentes avec ce que le modèle a appris.

> **Ce que ça veut dire :** Un modèle de machine learning est une formule mathématique. Il ne peut travailler qu'avec des chiffres. L'encodage est obligatoire, et la sauvegarde des encodeurs est une garantie de cohérence entre l'entraînement et la production.

---

## 5. Split Train / Test (temporel — respecte la chronologie)

In [ ]:
# Split temporel : train 2023+2024, test 2025
df_train = df[df["year"] <= 2024].copy()
df_test  = df[df["year"] == 2025].copy()

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]
X_test  = df_test[FEATURE_COLS]
y_test  = df_test[TARGET_COL]

print(f"Train : {X_train.shape[0]} lignes ({df_train['Date'].min().date()} → {df_train['Date'].max().date()})")
print(f"Test  : {X_test.shape[0]} lignes ({df_test['Date'].min().date()}  → {df_test['Date'].max().date()})")
print(f"Ratio : {len(X_test)/len(df)*100:.1f}% test")

## 6. Vérification de la distribution Train vs Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_train, bins=40, color="#2563EB", alpha=0.7, label="Train")
axes[0].hist(y_test, bins=40, color="#F59E0B", alpha=0.7, label="Test")
axes[0].set_title("Distribution de Sales — Train vs Test")
axes[0].set_xlabel("Sales")
axes[0].legend()

# Boxplot par Store_Type
df_plot = df.copy()
df_plot["Split"] = df_plot["year"].apply(lambda y: "Train" if y <= 2024 else "Test")
df_plot.boxplot(column="Sales", by="Store_Type", ax=axes[1])
axes[1].set_title("Sales par Store_Type")
plt.suptitle("")
plt.tight_layout()
plt.savefig("../reports/distribution_train_test.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé → reports/distribution_train_test.png")

## 7. Sauvegarde des données préparées

---
## ✅ Conclusion — Split Train / Test temporel

On découpe les données en deux parties :

```
2023 + 2024  →  3 726 lignes  (67.7%)  →  TRAIN  (le modèle apprend ici)
2025         →  1 774 lignes  (32.3%)  →  TEST   (on évalue ici)
```

On choisi un **split temporel** plutôt qu'un split aléatoire. Dans un problème de prévision, le modèle doit toujours apprendre sur le passé et prédire le futur — jamais l'inverse. Un split aléatoire laisserait le modèle "voir le futur" pendant l'entraînement, ce qui donnerait des résultats artificiellement bons mais inutilisables en production.

> **Ce que ça veut dire :** Ce découpage simule exactement les conditions d'utilisation réelle. Le modèle n'a jamais vu les données de 2025 pendant son entraînement. Quand il prédit sur 2025, c'est un vrai test en conditions réelles.

---

In [ ]:
# Sauvegarde pour reproductibilité
df.to_csv("../data/data_prepared.csv", index=False)

# Sauvegarde de la liste des features (utile pour l'API)
with open(os.path.join(MODELS_PATH, "feature_cols.pkl"), "wb") as f:
    pickle.dump(FEATURE_COLS, f)

print("✅ Données préparées sauvegardées → data/data_prepared.csv")
print("✅ Feature columns sauvegardées  → models/feature_cols.pkl")
print()
print("📊 Résumé Train :")
print(y_train.describe().round(2))
print("\n📊 Résumé Test :")
print(y_test.describe().round(2))

---
## ✅ Conclusion — Vérification des distributions

Les distributions Train et Test sont similaires :

- **Train** : Sales moyenne ≈ 9 850 FCFA
- **Test** : Sales moyenne ≈ 9 940 FCFA

Aucun décalage significatif entre les deux périodes. Les trois types de magasins (A, B, C) sont bien représentés dans les deux parties.

> **Ce que ça veut dire :** Le split est équilibré. Si la distribution du test était très différente du train, le modèle serait évalué sur des cas qu'il n'a jamais vus — ce qui fausserait les métriques. Ici, les deux distributions sont cohérentes, ce qui garantit une évaluation fiable.

---